# 🦺 PPE Detection — Hard Hat, Head, Person
**Модель:** YOLOv8n (nano — быстрая, лёгкая)  
**Классы:** Head · Helmet · Person  
**Датасет:** Roboflow (donjuvns-workspace)  

### Как запустить:
1. Меню → **Runtime → Change runtime type → T4 GPU**
2. Вставь свой API ключ в ячейку ниже
3. **Runtime → Run all**

## 1. Установка библиотек

In [ ]:
# Установка всех нужных библиотек
!pip install ultralytics roboflow --quiet

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA доступен: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Загрузка датасета с Roboflow

In [ ]:
from roboflow import Roboflow

# ⬇️ ВСТАВЬ СЮДА СВОЙ API КЛЮЧ
API_KEY = "ВСТАВЬ_API_КЛЮЧ_СЮДА"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("donjuvns-workspace").project("hard-hat-detection-62dxm-rpuma")

# Скачиваем последнюю версию в формате YOLOv8
versions = project.versions()
latest = versions[0].version if versions else 1
print(f'Скачиваем версию: {latest}')

dataset = project.version(latest).download("yolov8")
print(f'Датасет сохранён в: {dataset.location}')

## 3. Проверка датасета

In [ ]:
import os
import yaml
from pathlib import Path

# Находим data.yaml
yaml_path = Path(dataset.location) / 'data.yaml'
with open(yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

print('=== Конфигурация датасета ===')
print(f"Классы ({data_cfg['nc']}): {data_cfg['names']}")

# Считаем изображения в каждом сплите
for split in ['train', 'valid', 'test']:
    img_dir = Path(dataset.location) / split / 'images'
    if img_dir.exists():
        count = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
        print(f'{split}: {count} изображений')

In [ ]:
# Визуализация примеров из датасета
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

def show_samples(dataset_path, split='train', n=6):
    img_dir = Path(dataset_path) / split / 'images'
    lbl_dir = Path(dataset_path) / split / 'labels'
    
    images = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    samples = random.sample(images, min(n, len(images)))
    
    colors = {'0': 'red', '1': 'lime', '2': 'cyan'}
    class_names = data_cfg['names']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, img_path in enumerate(samples):
        img = Image.open(img_path)
        W, H = img.size
        axes[i].imshow(img)
        
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, cx, cy, w, h = parts
                    cx, cy, w, h = float(cx)*W, float(cy)*H, float(w)*W, float(h)*H
                    x1, y1 = cx - w/2, cy - h/2
                    color = colors.get(cls, 'yellow')
                    rect = patches.Rectangle((x1,y1), w, h,
                                             linewidth=2, edgecolor=color, facecolor='none')
                    axes[i].add_patch(rect)
                    axes[i].text(x1, y1-5, class_names[int(cls)],
                                 color=color, fontsize=9, fontweight='bold')
        axes[i].axis('off')
        axes[i].set_title(img_path.name, fontsize=8)
    
    plt.suptitle(f'Примеры из {split} выборки', fontsize=14)
    plt.tight_layout()
    plt.show()

show_samples(dataset.location)

## 4. Обучение модели YOLOv8

In [ ]:
from ultralytics import YOLO

# Загружаем предобученную YOLOv8n (nano — быстрее всего на Colab T4)
model = YOLO('yolov8n.pt')

print('Модель загружена. Начинаем обучение...')
print(f'Датасет: {yaml_path}')

In [ ]:
# ОБУЧЕНИЕ
results = model.train(
    data=str(yaml_path),
    epochs=50,          # 50 эпох — хватит для хорошего результата
    imgsz=640,          # стандартный размер YOLOv8
    batch=16,           # размер батча для T4 GPU
    patience=10,        # early stopping если нет улучшений
    name='ppe_hardhat', # имя эксперимента
    project='runs',     # папка для сохранения
    device=0,           # GPU
    workers=2,
    verbose=True,
    plots=True,         # сохранять графики
    save=True,          # сохранять лучшую модель
)

print('\n✅ Обучение завершено!')
print(f'Лучшая модель: runs/ppe_hardhat/weights/best.pt')

## 5. Оценка модели (Validation)

In [ ]:
# Загружаем лучшую обученную модель
best_model = YOLO('runs/ppe_hardhat/weights/best.pt')

# Валидация на тестовой выборке
metrics = best_model.val(data=str(yaml_path), split='test')

print('\n=== Результаты ===')
print(f'mAP@50:      {metrics.box.map50:.3f}  (цель: > 0.70)')
print(f'mAP@50-95:   {metrics.box.map:.3f}')
print(f'Precision:   {metrics.box.mp:.3f}')
print(f'Recall:      {metrics.box.mr:.3f}')

print('\nПо классам:')
for i, name in enumerate(data_cfg['names']):
    print(f'  {name}: mAP50 = {metrics.box.maps[i]:.3f}')

In [ ]:
# Показываем графики обучения
from IPython.display import Image as IPImage, display
import glob

plot_files = glob.glob('runs/ppe_hardhat/*.png')
for plot in sorted(plot_files):
    print(f'--- {Path(plot).name} ---')
    display(IPImage(plot, width=800))

## 6. Инференс — тест на изображениях

In [ ]:
import cv2
import numpy as np
from pathlib import Path

# Цвета для каждого класса
CLASS_COLORS = {
    'Head':    (255, 100, 100),   # красный — голова без каски
    'Helmet':  (100, 255, 100),   # зелёный — каска есть
    'Person':  (100, 200, 255),   # голубой — человек
}

def predict_and_show(img_path, conf=0.35):
    results = best_model.predict(img_path, conf=conf, verbose=False)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    counts = {name: 0 for name in data_cfg['names']}
    
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_id = int(box.cls[0])
            cls_name = data_cfg['names'][cls_id]
            conf_val = float(box.conf[0])
            color = CLASS_COLORS.get(cls_name, (255, 255, 0))
            counts[cls_name] += 1
            
            cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
            label = f'{cls_name} {conf_val:.2f}'
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(img, (x1, y1-th-8), (x1+tw+4, y1), color, -1)
            cv2.putText(img, label, (x1+2, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)
    
    # Итог: сколько без каски
    no_helmet = counts['Head']  # Head = голова без каски
    status = '🔴 НАРУШЕНИЕ' if no_helmet > 0 else '🟢 ВСЕ В КАСКАХ'
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.title(f'{status} | Helmet: {counts["Helmet"]}  Head (без каски): {counts["Head"]}  Person: {counts["Person"]}',
              fontsize=13)
    plt.axis('off')
    plt.show()

# Тест на нескольких тестовых изображениях
test_images = list((Path(dataset.location) / 'test' / 'images').glob('*.jpg'))[:4]
for img_path in test_images:
    predict_and_show(str(img_path))

## 7. Сохранение модели — скачать на компьютер

In [ ]:
import shutil
from google.colab import files

# Копируем лучшую модель
shutil.copy('runs/ppe_hardhat/weights/best.pt', '/content/ppe_best.pt')

# Скачиваем на компьютер
files.download('/content/ppe_best.pt')
print('✅ Файл ppe_best.pt скачан!')

## 8. (Бонус) Запуск на веб-камере локально
После скачивания `ppe_best.pt` — запусти `webcam_demo.py` на своём компьютере.